# SARIMA store-level baseline

Этот ноутбук перенесен в рефакторинг-поток как финальный статистический baseline на уровне `(store_nbr, family)`. Его область применения: сравнение на `top_pairs`, без обязательного полного submission на всех рядах.


In [2]:
import os
import time
import logging
from datetime import datetime

import numpy as np
import pandas as pd
import pmdarima as pm

from concurrent.futures import ProcessPoolExecutor, as_completed
import warnings

warnings.filterwarnings("ignore")

# Настройка отображения pandas
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# Настройка логгера
logger = logging.getLogger("sarima_store_level")
logger.setLevel(logging.INFO)

# если в ноутбуке логгер уже был настроен, не дублируем хендлеры
if not logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter(
        "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.info("Logger sarima_store_level initialized")


2025-11-16 19:23:07,684 - sarima_store_level - INFO - Logger sarima_store_level initialized


## Загрузка данных и первичная проверка

В качестве исходных данных используем уже предобработанные таблицы:

- `train_processed` — обучающая выборка с признаками и целевой переменной `sales`;
- `test_processed` — тестовая выборка в том же формате (без `sales`).

В этом ноутбуке будем работать только с колонками, необходимыми для SARIMA
на уровне `(store_nbr, family, date)`:

- `date` — дата (тип `datetime64[ns]`);
- `store_nbr` — идентификатор магазина;
- `family` — товарная категория;
- `sales` — целевая переменная.


In [3]:
DATA_DIR = "../data"

train_path = os.path.join(DATA_DIR, "train_processed.csv")
test_path  = os.path.join(DATA_DIR, "test_processed.csv")

logger.info(f"Loading train from {train_path}")
train_processed = pd.read_csv(train_path, parse_dates=["date"])

logger.info(f"Loading test from {test_path}")
test_processed = pd.read_csv(test_path, parse_dates=["date"])

train_processed.shape, test_processed.shape

2025-11-16 19:23:07,700 - sarima_store_level - INFO - Loading train from ../data/train_processed.csv
2025-11-16 19:23:24,170 - sarima_store_level - INFO - Loading test from ../data/test_processed.csv


((3008016, 21), (28512, 15))

In [4]:
logger.info("Train head:")
display(train_processed.head())

logger.info("Train info:")
print(train_processed.dtypes)

logger.info("Test head:")
display(test_processed.head())

logger.info("Test dtypes:")
print(test_processed.dtypes)

# Проверка пропусков в ключевых колонках
key_cols = ["date", "store_nbr", "family", "sales"]
na_train = train_processed[key_cols].isna().sum()
logger.info("NaN counts in train (key columns):")
print(na_train)

key_cols_test = ["date", "store_nbr", "family"]
na_test = test_processed[key_cols_test].isna().sum()
logger.info("NaN counts in test (key columns):")
print(na_test)

# Диапазон дат
logger.info(
    f"Train dates: {train_processed['date'].min().date()} "
    f"→ {train_processed['date'].max().date()}"
)
logger.info(
    f"Test dates:  {test_processed['date'].min().date()} "
    f"→ {test_processed['date'].max().date()}"
)

2025-11-16 19:23:24,259 - sarima_store_level - INFO - Train head:


,date,store_nbr,family,sales,onpromotion,city,state,store_type,cluster,dcoilwtico,day_of_week,type,locale,locale_name,description,transferred,agg_description,is_holiday,transactions,sales_sum,is_payday
0,2013-01-01,1,AUTOMOTIVE,0.0,0.0,Quito,Pichincha,D,13,93.14,Tuesday,Holiday,National,Ecuador,Primer dia del ano,False,Праздник,True,0.0,0.0,0
1,2013-01-01,1,BABY CARE,0.0,0.0,Quito,Pichincha,D,13,93.14,Tuesday,Holiday,National,Ecuador,Primer dia del ano,False,Праздник,True,0.0,0.0,0
2,2013-01-01,1,BEAUTY,0.0,0.0,Quito,Pichincha,D,13,93.14,Tuesday,Holiday,National,Ecuador,Primer dia del ano,False,Праздник,True,0.0,0.0,0
3,2013-01-01,1,BEVERAGES,0.0,0.0,Quito,Pichincha,D,13,93.14,Tuesday,Holiday,National,Ecuador,Primer dia del ano,False,Праздник,True,0.0,0.0,0
4,2013-01-01,1,BOOKS,0.0,0.0,Quito,Pichincha,D,13,93.14,Tuesday,Holiday,National,Ecuador,Primer dia del ano,False,Праздник,True,0.0,0.0,0


2025-11-16 19:23:24,287 - sarima_store_level - INFO - Train info:
2025-11-16 19:23:24,289 - sarima_store_level - INFO - Test head:


date               datetime64[ns]
store_nbr                   int64
family                     object
sales                     float64
onpromotion               float64
city                       object
state                      object
store_type                 object
cluster                     int64
dcoilwtico                float64
day_of_week                object
type                       object
locale                     object
locale_name                object
description                object
transferred                  bool
agg_description            object
is_holiday                   bool
transactions              float64
sales_sum                 float64
is_payday                   int64
dtype: object


,id,date,store_nbr,family,onpromotion,city,state,store_type,cluster,type,description,transferred,agg_description,is_holiday,is_payday
0,3000888,2017-08-16,1,AUTOMOTIVE,0,Quito,Pichincha,D,13,Work Day,None,False,Другое,0,0
1,3000889,2017-08-16,1,BABY CARE,0,Quito,Pichincha,D,13,Work Day,None,False,Другое,0,0
2,3000890,2017-08-16,1,BEAUTY,2,Quito,Pichincha,D,13,Work Day,None,False,Другое,0,0
3,3000891,2017-08-16,1,BEVERAGES,20,Quito,Pichincha,D,13,Work Day,None,False,Другое,0,0
4,3000892,2017-08-16,1,BOOKS,0,Quito,Pichincha,D,13,Work Day,None,False,Другое,0,0


2025-11-16 19:23:24,302 - sarima_store_level - INFO - Test dtypes:


id                          int64
date               datetime64[ns]
store_nbr                   int64
family                     object
onpromotion                 int64
city                       object
state                      object
store_type                 object
cluster                     int64
type                       object
description                object
transferred                  bool
agg_description            object
is_holiday                  int64
is_payday                   int64
dtype: object


2025-11-16 19:23:24,650 - sarima_store_level - INFO - NaN counts in train (key columns):
2025-11-16 19:23:24,659 - sarima_store_level - INFO - NaN counts in test (key columns):
2025-11-16 19:23:24,672 - sarima_store_level - INFO - Train dates: 2013-01-01 → 2017-08-15
2025-11-16 19:23:24,673 - sarima_store_level - INFO - Test dates:  2017-08-16 → 2017-08-31


date         0
store_nbr    0
family       0
sales        0
dtype: int64
date         0
store_nbr    0
family       0
dtype: int64


In [5]:
logger.info(
    f"Unique stores in train: {train_processed['store_nbr'].nunique()}, "
    f"families: {train_processed['family'].nunique()}"
)

2025-11-16 19:23:24,808 - sarima_store_level - INFO - Unique stores in train: 54, families: 33


## Временные сплиты для оценки SARIMA

Для оценки качества SARIMA будем использовать временные сплиты (time-based CV):

- **формат**: expanding window (train-окно растёт, валид. окно фиксированное),
- **горизонт валидации**: 16 дней (как в тесте соревнования),
- **количество сплитов**: 3 (пример: `split_1`, `split_2`, `split_3`).

На каждом сплите:
- SARIMA обучается только на историческом train-окне,
- прогноз строится на следующие 16 дней,
- метрики считаются на уровне `(store_nbr, family, date)`.

Далее определим функцию, которая по `train_processed` вернёт список таких сплитов.


In [6]:
def generate_time_splits(
    df,
    n_splits=3,
    horizon_days=16,
    step_days=30,
    date_col="date",
    last_val_end=None,
):
    """
    Генерирует временные сплиты вида expanding window с фиксированным горизонтом валидации.

    Параметры:
    - df: DataFrame с колонкой дат.
    - n_splits: сколько сплитов строим.
    - horizon_days: длина окна валидации в днях (для Favorita = 16).
    - step_days: на сколько дней сдвигаем валидационное окно назад между сплитами.
    - date_col: имя колонки с датой.
    - last_val_end: последняя дата, которую можно использовать как конец валидации
      (если None — берём максимальную дату из df).

    Возвращает:
    - список словарей с полями:
      {
        "name": str,
        "train_idx": Index,
        "val_idx": Index,
        "train_end": Timestamp,
        "val_start": Timestamp,
        "val_end": Timestamp,
      }
    """
    df = df.sort_values(date_col).copy()
    unique_dates = df[date_col].drop_duplicates().sort_values().to_numpy()

    if last_val_end is None:
        last_val_end = pd.to_datetime(unique_dates[-1])
    else:
        last_val_end = pd.to_datetime(last_val_end)

    splits = []

    for k in range(n_splits):
        # конец и начало валидационного окна
        val_end = last_val_end - pd.Timedelta(days=step_days * k)
        val_start = val_end - pd.Timedelta(days=horizon_days - 1)

        # конец обучающего периода
        train_end = val_start - pd.Timedelta(days=1)

        train_mask = df[date_col] <= train_end
        val_mask = (df[date_col] >= val_start) & (df[date_col] <= val_end)

        train_idx = df.index[train_mask]
        val_idx = df.index[val_mask]

        # если вдруг окно вылезло за границы — пропускаем
        if len(train_idx) == 0 or len(val_idx) == 0:
            continue

        splits.append(
            {
                "name": f"split_{k+1}",
                "train_idx": train_idx,
                "val_idx": val_idx,
                "train_end": train_end,
                "val_start": val_start,
                "val_end": val_end,
            }
        )

    return splits

In [7]:
splits = generate_time_splits(
    train_processed,
    n_splits=3,
    horizon_days=16,
    step_days=30,
    date_col="date",
)

for s in splits:
    tr = train_processed.loc[s["train_idx"]]
    va = train_processed.loc[s["val_idx"]]
    print(
        f'{s["name"]} | '
        f'train: {tr["date"].min()} → {tr["date"].max()} | n = {len(tr)} | '
        f'val: {va["date"].min()} → {va["date"].max()} | n = {len(va)}'
    )


split_1 | train: 2013-01-01 00:00:00 → 2017-07-30 00:00:00 | n = 2979504 | val: 2017-07-31 00:00:00 → 2017-08-15 00:00:00 | n = 28512
split_2 | train: 2013-01-01 00:00:00 → 2017-06-30 00:00:00 | n = 2926044 | val: 2017-07-01 00:00:00 → 2017-07-16 00:00:00 | n = 28512
split_3 | train: 2013-01-01 00:00:00 → 2017-05-31 00:00:00 | n = 2872584 | val: 2017-06-01 00:00:00 → 2017-06-16 00:00:00 | n = 28512


In [8]:
import json
from pathlib import Path

def make_splits_jsonable(splits):
    jsonable = []
    for s in splits:
        jsonable.append(
            {
                "name": s["name"],
                # индексы в список, чтобы были JSON‑серилизируемыми
                "train_idx": s["train_idx"].tolist(),
                "val_idx": s["val_idx"].tolist(),
                # даты в строку (можно isoformat, можно свой формат)
                "train_end": s["train_end"].strftime("%Y-%m-%d"),
                "val_start": s["val_start"].strftime("%Y-%m-%d"),
                "val_end": s["val_end"].strftime("%Y-%m-%d"),
            }
        )
    return jsonable

# генерируем сплиты
splits = generate_time_splits(
    train_processed,
    n_splits=3,
    horizon_days=16,
    step_days=30,
    date_col="date",
)

# делаем структуру, пригодную для JSON
splits_json = make_splits_jsonable(splits)

# путь к файлу
out_dir = Path("../data/experiment_info")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "splits.json"

# сохранение
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(splits_json, f, indent=2, ensure_ascii=False)

logger.info(f"Сохранено в {out_path}")

2025-11-16 19:23:40,460 - sarima_store_level - INFO - Сохранено в ../data/experiment_info/splits.json


## Выбор репрезентативного подмножества серий

Обучать и оценивать SARIMA на всех `(store_nbr, family)` (1782 рядов)
слишком затратно по времени. Вместо этого:

- оцениваем **важность** пары `(store_nbr, family)` по суммарным продажам
  за весь train;
- выбираем топ-20% пар по этому показателю — такие серии покрывают
  основную долю оборота и хорошо отражают бизнес-значимые ряды;
- далее все эксперименты с SARIMA (и CatBoost в отдельном ноутбуке) будем
  проводить на этом фиксированном подмножестве серий.

Это уменьшает время экспериментов, но остаётся методологически корректным:
мы явно работаем с «верхушкой» по выручке и используем одинаковый
подход к выбору серий для всех моделей.


In [9]:
# Считаем суммарные продажи по (store_nbr, family)
series_sales = (
    train_processed
    .groupby(["store_nbr", "family"])["sales"]
    .sum()
    .rename("sales_sum")
    .reset_index()
)

logger.info(f"Total number of (store, family) series: {len(series_sales)}")

# Считаем долю каждой серии в общем обороте
total_sales = series_sales["sales_sum"].sum()
series_sales["sales_share"] = series_sales["sales_sum"] / total_sales

# Берём топ-20% серий по сумме продаж
quantile_cut = series_sales["sales_sum"].quantile(0.8)
top_series = series_sales[series_sales["sales_sum"] >= quantile_cut].copy()

logger.info(
    f"Selected {len(top_series)} top series (>= 80th percentile by sales_sum)"
)

# Доля оборота, покрываемая выбранными сериями
top_sales_share = top_series["sales_sum"].sum() / total_sales
logger.info(
    f"Top series cover {top_sales_share:.3%} of total sales"
)

top_series.head()


2025-11-16 19:23:40,736 - sarima_store_level - INFO - Total number of (store, family) series: 1782
2025-11-16 19:23:40,843 - sarima_store_level - INFO - Selected 357 top series (>= 80th percentile by sales_sum)
2025-11-16 19:23:40,844 - sarima_store_level - INFO - Top series cover 88.598% of total sales


,store_nbr,family,sales_sum,sales_share
3,1,BEVERAGES,2.673769e+06,0.002490
5,1,BREAD/BAKERY,5.699922e+05,0.000531
7,1,CLEANING,1.078525e+06,0.001005
8,1,DAIRY,1.054354e+06,0.000982
12,1,GROCERY I,3.743823e+06,0.003487


In [10]:
# Множество пар (store_nbr, family), с которыми будем работать
top_pairs = set(
    top_series[["store_nbr", "family"]].itertuples(index=False, name=None)
)

logger.info(f"Total number of (store, family) series: {len(top_pairs)}")

def filter_to_top_pairs(df):
    """
    Возвращает подтаблицу df только по выбранным top_pairs (store_nbr, family).
    """
    mask = df[["store_nbr", "family"]].apply(tuple, axis=1).isin(top_pairs)
    return df[mask].copy()


2025-11-16 19:23:40,916 - sarima_store_level - INFO - Total number of (store, family) series: 357


In [11]:
# Проверим, сколько наблюдений остаётся на одном из сплитов
s = splits[0]
train_df_split = train_processed.loc[s["train_idx"]]
val_df_split = train_processed.loc[s["val_idx"]]

train_df_top = filter_to_top_pairs(train_df_split)
val_df_top   = filter_to_top_pairs(val_df_split)

logger.info(
    f"{s['name']} | "
    f"train full: {len(train_df_split)}, train top_pairs: {len(train_df_top)}; "
    f"val full: {len(val_df_split)}, val top_pairs: {len(val_df_top)}"
)

train_df_top[["store_nbr", "family"]].drop_duplicates().head()

2025-11-16 19:23:53,878 - sarima_store_level - INFO - split_1 | train full: 2979504, train top_pairs: 596904; val full: 28512, val top_pairs: 5712


,store_nbr,family
2061060,38,GROCERY I
2061056,38,DAIRY
2061055,38,CLEANING
2061053,38,BREAD/BAKERY
2061051,38,BEVERAGES


Это нормальный и методологически чистый срез:

- по количеству данных он достаточно большой, чтобы SARIMA раскрылась;

- по времени он заметно дешевле 100% набора, при этом покрывает самый важные для бизнеса комбинации товар x категория.

## Ядро SARIMA для одного ряда (store, family)

Далее реализуем вспомогательные функции для работы с SARIMA:

- `get_series_for_pair_store` — строит временной ряд продаж для пары `(store_nbr, family)`;
- `fallback_forecast_recursive` — быстрый эвристический прогноз на случай, когда SARIMA
  не подходит или падает;
- `fit_sarima_one_series` — обучение одной SARIMA-модели для ряда;
- `forecast_sarima` — получение прогноза с горизонтом 16 дней и защитой от ошибок.

Важно: здесь мы работаем на уровне `(store_nbr, family, date)` без агрегации до кластеров
и без top-down. Все метрики позже будут считаться на этом же уровне.


In [12]:
def get_series_for_pair_store(train_df, store_nbr, family):
    """
    Возвращает временной ряд продаж для пары (store_nbr, family).

    Формат:
    - индекс: date (отсортирован)
    - значения: daily sales (float32)
    """
    sub = train_df[
        (train_df["store_nbr"] == store_nbr) &
        (train_df["family"] == family)
    ]

    if sub.empty:
        return pd.Series(dtype="float32")

    ts = (
        sub.groupby("date")["sales"]
        .sum()   # или .mean(), но для одного магазина/семьи sum == исходным продажам
        .sort_index()
        .astype("float32")
    )
    return ts

In [13]:
def fallback_forecast_recursive(ts: pd.Series, horizon: int = 16) -> np.ndarray:
    """
    Простой рекурсивный эвристический прогноз на horizon дней вперёд.

    Идея:
    - медиана последних 30 значений (история + уже спрогнозированные);
    - уровень недели назад или последнее значение;
    - прогноз = среднее этих двух оценок, обрезанное снизу нулём.
    """
    ts = pd.Series(ts).sort_index().astype("float64")

    if ts.empty:
        return np.zeros(horizon, dtype="float32")

    values = ts.to_numpy().tolist()
    y_hat = np.empty(horizon, dtype="float32")

    for h in range(horizon):
        window = values[-30:]
        med_30 = float(np.nanmedian(window))

        if len(values) >= 7:
            seasonal_level = values[-7]
        else:
            seasonal_level = values[-1]

        base = 0.5 * med_30 + 0.5 * seasonal_level
        base = 0.0 if not np.isfinite(base) else max(base, 0.0)

        y_hat[h] = base
        values.append(base)

    return y_hat


In [14]:
def fit_sarima_one_series(y_train: pd.Series):
    """
    Обучение SARIMA (auto_arima, pmdarima) для одного ряда продаж.

    Параметры подобраны консервативно, чтобы не взорвать время.
    """
    logger = logging.getLogger("sarima_store_level")

    y = pd.Series(y_train).astype("float64").fillna(0.0).clip(lower=0.0)

    # минимальные требования к длине ряда
    if len(y) < 30 or y.sum() == 0:
        logger.info("Series too short or zero-sum, skipping SARIMA")
        return None

    if y.isna().any():
        logger.warning(
            f"Series has NaN after preprocessing: {y.isna().sum()} values"
        )
        return None

    try:
        model = pm.auto_arima(
            y,
            seasonal=True,
            m=7,              # недельная сезонность
            start_p=0, max_p=2,
            start_q=0, max_q=2,
            d=None,           # авто-выбор d
            start_P=0, max_P=1,
            start_Q=0, max_Q=1,
            D=None,           # авто-выбор D
            error_action="ignore",
            suppress_warnings=True,
            stepwise=True,
            max_order=5,
            trace=False,
        )
        logger.info(
            f"auto_arima fitted: order={model.order}, "
            f"seasonal_order={model.seasonal_order}"
        )
        return model
    except Exception as e:
        logger.warning(f"auto_arima failed: {e}")
        return None


In [15]:
def forecast_sarima(model, horizon: int = 16) -> np.ndarray:
    """
    Возвращает прогноз на horizon дней вперёд для одной серии.

    Если model is None или predict падает, возвращает нули.
    """
    if model is None:
        return np.zeros(horizon, dtype="float32")

    try:
        y_forecast = model.predict(n_periods=horizon)
        y_forecast = np.array(y_forecast, dtype="float32")
        y_forecast = np.clip(y_forecast, 0.0, None)
        return y_forecast
    except Exception as e:
        logger = logging.getLogger("sarima_store_level")
        logger.warning(f"SARIMA predict failed: {e}")
        return None


## Параллельный прогон SARIMA по (store, family)

Чтобы ускорить обучение большого числа SARIMA-моделей, будем:

- инициализировать воркеры с помощью `ProcessPoolExecutor`, передавая им `train_df`;
- для каждой пары `(store_nbr, family)` вызывать функцию `run_sarima_for_pair_store`,
  которая строит ряд, решает, применять ли SARIMA или фолбэк, и возвращает прогноз
  на валидационное окно;
- собирать все прогнозы в одном DataFrame уровня `(date, store_nbr, family, y_pred)`.

Функция `predict_sarima_on_split` в таком дизайне отвечает **только** за:
- выбор пар и формирование задач,
- параллельный запуск,
- сбор прогнозов.

Подсчёт метрик будет вынесен в отдельную функцию.


In [16]:
GLOBAL_TRAIN_DF = None

def init_worker(train_df):
    """
    Инициализатор воркера: сохраняем train_df в глобальной переменной,
    чтобы не передавать тяжёлый DataFrame в каждый submit.
    """
    global GLOBAL_TRAIN_DF
    GLOBAL_TRAIN_DF = train_df
    logger = logging.getLogger("sarima_store_level")
    logger.info("Worker initialized with GLOBAL_TRAIN_DF")

In [17]:
def run_sarima_for_pair_store(store_nbr, family, val_start, val_end):
    """
    Строит SARIMA-прогноз для одной пары (store_nbr, family) на валидационном окне.

    Возвращает DataFrame:
    - date
    - store_nbr
    - family
    - y_pred
    длиной = числу дней в интервале [val_start, val_end].
    """
    logger = logging.getLogger("sarima_store_level")

    global GLOBAL_TRAIN_DF
    if GLOBAL_TRAIN_DF is None:
        raise RuntimeError("GLOBAL_TRAIN_DF is not initialized in worker")

    pid = os.getpid()

    ts_train = get_series_for_pair_store(GLOBAL_TRAIN_DF, store_nbr, family)

    horizon = (val_end - val_start).days + 1
    dates = pd.date_range(val_start, val_end, freq="D")

    # если ряд пустой/слишком короткий/нулевой — сразу фолбэк
    if ts_train.empty or len(ts_train) < 10 or ts_train.sum() == 0:
        logger.info(
            f"[PID={pid}] fallback for (store={store_nbr}, family={family}): "
            f"empty/short/zero series"
        )
        y_hat = fallback_forecast_recursive(ts_train, horizon=horizon)
    else:
        logger.info(
            f"[PID={pid}] start (store={store_nbr}, family={family})"
        )
        try:
            model = fit_sarima_one_series(ts_train)
            if model is None:
                y_hat = fallback_forecast_recursive(ts_train, horizon=horizon)
                logger.info(
                    f"[PID={pid}] SARIMA skipped, using fallback for "
                    f"(store={store_nbr}, family={family})"
                )
            else:
                y_hat = forecast_sarima(model, horizon=horizon)
                
                if y_hat is None:
                    y_hat = fallback_forecast_recursive(ts_train, horizon=horizon)
                    logger.info(
                        f"[PID={pid}] SARIMA skipped, using fallback for "
                        f"(store={store_nbr}, family={family})"
                    )
                else:
                    logger.info(
                        f"[PID={pid}] SARIMA OK for (store={store_nbr}, family={family}), "
                        f"horizon={horizon}"
                    )
        except Exception as e:
            logger.warning(
                f"[PID={pid}] run_sarima_for_pair_store failed for "
                f"(store={store_nbr}, family={family}): {e}. Using fallback."
            )
            y_hat = fallback_forecast_recursive(ts_train, horizon=horizon)

    df_pred = pd.DataFrame({
        "date": dates,
        "store_nbr": store_nbr,
        "family": family,
        "y_pred": y_hat.astype("float32"),
    })

    return df_pred


In [18]:
def predict_sarima_on_split(
    train_df,
    val_df,
    split_name,
    max_workers=4,
    pairs_subset=None,
):
    """
    Строит SARIMA-прогнозы для всех (или выбранных) пар (store_nbr, family)
    на заданном сплите.

    Параметры:
    - train_df: train-часть сплита
    - val_df:   val-часть сплита
    - split_name: имя сплита для логов
    - max_workers: число воркеров в ProcessPoolExecutor
    - pairs_subset: опциональный список/сет пар (store_nbr, family),
      если хотим ограничиться подмножеством.

    Возвращает:
    - preds_df: DataFrame с колонками [date, store_nbr, family, y_pred]
      на всём валидационном окне.
    """
    logger = logging.getLogger("sarima_store_level")

    split_start_time = time.time()
    split_start_dt = datetime.now()
    logger.info(
        f"[{split_name}] Start SARIMA prediction on split "
        f"at {split_start_dt.strftime('%Y-%m-%d %H:%M:%S')}"
    )

    val_start = val_df["date"].min()
    val_end = val_df["date"].max()
    logger.info(
        f"[{split_name}] Validation window: {val_start.date()} → {val_end.date()}"
    )

    # полный список пар в train
    pairs = (
        train_df[["store_nbr", "family"]]
        .drop_duplicates()
        .itertuples(index=False, name=None)
    )
    pairs = list(pairs)

    # если передан pairs_subset, фильтруем список пар
    if pairs_subset is not None:
        pairs_set = set(pairs_subset)
        pairs = [p for p in pairs if p in pairs_set]

    n_pairs = len(pairs)
    logger.info(f"[{split_name}] Number of (store, family) pairs: {n_pairs}")

    results = []
    n_ok = 0
    n_fail = 0
    total_pair_time = 0.0

    logger.info(
        f"[{split_name}] Launching ProcessPoolExecutor with max_workers={max_workers}"
    )

    with ProcessPoolExecutor(
        max_workers=max_workers,
        initializer=init_worker,
        initargs=(train_df,),
    ) as executor:
        future_to_meta = {}
        for (store_nbr, family) in pairs:
            submit_time = time.time()
            fut = executor.submit(
                run_sarima_for_pair_store,
                store_nbr,
                family,
                val_start,
                val_end,
            )
            future_to_meta[fut] = (store_nbr, family, submit_time)

        for i, future in enumerate(as_completed(future_to_meta), start=1):
            store_nbr, family, submit_time = future_to_meta[future]
            pair_start_time = submit_time

            try:
                df_pred = future.result()
                n_ok += 1
                pair_elapsed = time.time() - pair_start_time
                total_pair_time += pair_elapsed
                logger.info(
                    f"[{split_name}] Pair (store={store_nbr}, family={family}) "
                    f"finished in {pair_elapsed:.2f} seconds "
                    f"(queue + fit+forecast)"
                )
            except Exception as exc:
                n_fail += 1
                logger.warning(
                    f"[{split_name}] Failed pair (store={store_nbr}, family={family}): {exc}"
                )
                pair_elapsed = time.time() - pair_start_time
                total_pair_time += pair_elapsed
                logger.info(
                    f"[{split_name}] Failed pair (store={store_nbr}, family={family}) "
                    f"took {pair_elapsed:.2f} seconds before failure"
                )

                dates = pd.date_range(val_start, val_end, freq="D")
                df_pred = pd.DataFrame({
                    "date": dates,
                    "store_nbr": store_nbr,
                    "family": family,
                    "y_pred": np.zeros(len(dates), dtype="float32"),
                })

            results.append(df_pred)

            if i % 20 == 0 or i == n_pairs:
                avg_pair_time = total_pair_time / i
                logger.info(
                    f"[{split_name}] Progress: {i}/{n_pairs} pairs processed "
                    f"(ok={n_ok}, fail={n_fail}), "
                    f"avg pair time so far = {avg_pair_time:.2f} seconds "
                    f"(queue + fit+forecast)"
                )

    split_elapsed = time.time() - split_start_time
    avg_pair_time_overall = total_pair_time / max(n_pairs, 1)
    logger.info(
        f"[{split_name}] All pairs processed in {split_elapsed:.2f} seconds "
        f"(avg per pair = {avg_pair_time_overall:.2f} seconds, "
        f"queue + fit+forecast)"
    )

    preds_df = pd.concat(results, ignore_index=True)
    logger.info(f"[{split_name}] preds_df shape: {preds_df.shape}")

    return preds_df


## Расчёт метрик на уровне (store, family, date)

Чтобы разделить ответственность:

- функция `predict_sarima_on_split` строит прогнозы и возвращает DataFrame `preds_df`;
- отдельная функция `evaluate_on_split_store_level` принимает `preds_df` и `val_df`,
  мёрджит их, отфильтровывает пары с валидным `y_true` и считает метрики.

Так у нас чистая SRP:
- прогноз — это одна функция,
- метрики — другая,
- и легко переиспользовать каждую из них отдельно.


In [19]:
def calculate_metrics(y_true, y_pred):
    """
    Расчёт базовых метрик:
    - RMSLE
    - RMSE
    - MAE
    - WAPE (weighted absolute percentage error)

    Прогноз обрезается снизу нулём, если пришёл отрицательным.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    y_pred = np.clip(y_pred, 0, None)

    # RMSLE
    rmsle = np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2))

    # RMSE
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    # MAE
    mae = np.mean(np.abs(y_true - y_pred))

    # WAPE
    denom = np.sum(np.abs(y_true))
    if denom == 0:
        wape = np.nan
    else:
        wape = np.sum(np.abs(y_true - y_pred)) / denom

    return {
        "RMSLE": rmsle,
        "RMSE": rmse,
        "MAE": mae,
        "WAPE": wape,
    }

In [20]:
def evaluate_on_split_store_level(preds_df, val_df, split_name):
    """
    Считает метрики на уровне (store_nbr, family, date) для заданного сплита.

    Параметры:
    - preds_df: DataFrame с колонками [date, store_nbr, family, y_pred],
      результат работы predict_sarima_on_split.
    - val_df: DataFrame валидационной части (исходный train_processed с колонкой sales).
    - split_name: имя сплита (для логов и результата).

    Возвращает:
    - metrics: dict с ключами RMSLE, RMSE, MAE, WAPE, split, model.
    - val_pred_df: DataFrame со слитыми y_true и y_pred для дальнейшего анализа.
    """
    logger = logging.getLogger("sarima_store_level")

    # Берём из val_df фактические продажи на уровне (date, store_nbr, family)
    # У нас в val_df уже есть колонка sales, просто переименуем её
    val_true = val_df[["date", "store_nbr", "family", "sales"]].copy()
    val_true = val_true.rename(columns={"sales": "y_true"})

    # Слияние
    val_pred_df = preds_df.merge(
        val_true,
        on=["date", "store_nbr", "family"],
        how="left",
    )

    before_filter = len(val_pred_df)
    val_pred_df = val_pred_df[~val_pred_df["y_true"].isna()].copy()
    after_filter = len(val_pred_df)

    logger.info(
        f"[{split_name}] val_pred_df filtered: {before_filter} → {after_filter} "
        f"rows with y_true"
    )

    y_true = val_pred_df["y_true"].values
    y_pred = val_pred_df["y_pred"].values

    metrics = calculate_metrics(y_true, y_pred)
    metrics["split"] = split_name
    metrics["model"] = "sarima_store_family"

    logger.info(
        f"[{split_name}] Metrics: "
        f"RMSLE={metrics['RMSLE']:.4f}, "
        f"RMSE={metrics['RMSE']:.2f}, "
        f"MAE={metrics['MAE']:.2f}, "
        f"WAPE={metrics['WAPE']:.4f}"
    )

    return metrics, val_pred_df

## Тестовый прогон SARIMA на одном сплите

Перед тем как запускать все сплиты, проверим цепочку на одном сплите
(например, `split_1`) и ограниченном количестве серий из `top_pairs`
(например, 40–80 пар `(store_nbr, family)`).

Шаги:
1. Берём train/val для выбранного сплита и фильтруем по `top_pairs`.
2. Формируем небольшой `pairs_subset` из выбранных пар.
3. Запускаем `predict_sarima_on_split` для этого подмножества.
4. Считаем метрики через `evaluate_on_split_store_level`.
5. Сохраняем результаты в словари для последующего анализа.


In [22]:
# # Словари для хранения результатов
# sarima_metrics_by_split = {}
# sarima_preds_by_split = {}

# # Берём первый сплит
# s = splits[0]
# split_name = s["name"]

# # Train/val для сплита
# train_df_split = train_processed.loc[s["train_idx"]]
# val_df_split   = train_processed.loc[s["val_idx"]]

# # Фильтрация по top_pairs
# train_df_top = filter_to_top_pairs(train_df_split)
# val_df_top   = filter_to_top_pairs(val_df_split)

# logger.info(
#     f"{split_name} (top_pairs) | "
#     f"train: {len(train_df_top)} rows, "
#     f"val: {len(val_df_top)} rows"
# )

# # Список всех пар (store, family) в этом сплите после фильтрации
# pairs_all = (
#     train_df_top[["store_nbr", "family"]]
#     .drop_duplicates()
#     .itertuples(index=False, name=None)
# )
# pairs_all = list(pairs_all)
# logger.info(f"{split_name} (top_pairs) | total pairs: {len(pairs_all)}")

# # Ограничим тест, например, 40 парами
# N_TEST_PAIRS = 40
# pairs_subset = pairs_all[:N_TEST_PAIRS]
# logger.info(f"{split_name} (top_pairs) | using {len(pairs_subset)} pairs for test run")

# # Запуск SARIMA-прогноза
# MAX_WORKERS = 3  # для с1.32, yandex datasphere (32 ядра + 196 гб RAM)

# preds_df_test = predict_sarima_on_split(
#     train_df=train_df_top,
#     val_df=val_df_top,
#     split_name=split_name + "_test",
#     max_workers=MAX_WORKERS,
#     pairs_subset=pairs_subset,
# )

# # Расчёт метрик
# metrics_test, val_pred_df_test = evaluate_on_split_store_level(
#     preds_df_test,
#     val_df_top,
#     split_name=split_name + "_test",
# )

# # Сохраняем
# sarima_metrics_by_split[split_name + "_test"] = metrics_test
# sarima_preds_by_split[split_name + "_test"] = val_pred_df_test

# metrics_test

2025-11-16 18:52:27,341 - sarima_store_level - INFO - split_1 (top_pairs) | train: 596904 rows, val: 5712 rows
2025-11-16 18:52:27,387 - sarima_store_level - INFO - split_1 (top_pairs) | total pairs: 357
2025-11-16 18:52:27,388 - sarima_store_level - INFO - split_1 (top_pairs) | using 40 pairs for test run
2025-11-16 18:52:27,389 - sarima_store_level - INFO - [split_1_test] Start SARIMA prediction on split at 2025-11-16 18:52:27
2025-11-16 18:52:27,390 - sarima_store_level - INFO - [split_1_test] Validation window: 2017-07-31 → 2017-08-15
2025-11-16 18:52:27,435 - sarima_store_level - INFO - [split_1_test] Number of (store, family) pairs: 40
2025-11-16 18:52:27,436 - sarima_store_level - INFO - [split_1_test] Launching ProcessPoolExecutor with max_workers=3
2025-11-16 18:52:27,464 - sarima_store_level - INFO - Worker initialized with GLOBAL_TRAIN_DF
2025-11-16 18:52:27,480 - sarima_store_level - INFO - Worker initialized with GLOBAL_TRAIN_DF
2025-11-16 18:52:27,496 - sarima_store_level

{'RMSLE': 2.099106020893654,
 'RMSE': 1026.0805622158116,
 'MAE': 591.1401225275803,
 'WAPE': 0.30027957438383474,
 'split': 'split_1_test',
 'model': 'sarima_store_family'}

In [25]:
val_pred_df_test[["y_true", "y_pred"]].describe()

,y_true,y_pred
count,640.000000,640.000000
mean,1968.632478,1953.322632
std,2009.611224,2130.414795
min,108.000000,0.000000
25%,586.471250,509.719673
50%,1041.276500,921.840454
75%,2851.000000,3016.117310
max,11352.238000,10304.978516


## Полный прогон SARIMA по всем сплитам (top_pairs)

Теперь запускаем SARIMA на всех временных сплитах, используя:

- train/val окна из `splits`;
- фильтрацию по `top_pairs` (репрезентативные серии);
- `predict_sarima_on_split` для получения прогнозов;
- `evaluate_on_split_store_level` для расчёта метрик на уровне `(store, family, date)`.

Для каждого сплита:
- сохраняем метрики в `sarima_metrics_by_split`,
- сохраняем предсказания и факт в `sarima_preds_by_split`.

В конце соберём сводную таблицу метрик.


In [ ]:
sarima_metrics_by_split = {}
sarima_preds_by_split = {}

MAX_WORKERS = 3  # при необходимости скорректировать

for s in splits:
    split_name = s["name"]
    logger.info(f"=== {split_name} ===")

    # Train/val для сплита
    train_df_split = train_processed.loc[s["train_idx"]]
    val_df_split   = train_processed.loc[s["val_idx"]]

    # Фильтрация по top_pairs
    train_df_top = filter_to_top_pairs(train_df_split)
    val_df_top   = filter_to_top_pairs(val_df_split)

    logger.info(
        f"{split_name} (top_pairs) | "
        f"train: {len(train_df_top)} rows, "
        f"val: {len(val_df_top)} rows"
    )

    # Список пар для этого сплита (все top_pairs, которые реально есть в train)
    pairs_all = (
        train_df_top[["store_nbr", "family"]]
        .drop_duplicates()
        .itertuples(index=False, name=None)
    )
    pairs_all = list(pairs_all)
    logger.info(f"{split_name} (top_pairs) | total pairs: {len(pairs_all)}")

    # Прогон SARIMA по всем этим парам
    preds_df = predict_sarima_on_split(
        train_df=train_df_top,
        val_df=val_df_top,
        split_name=split_name,
        max_workers=MAX_WORKERS,
        pairs_subset=pairs_all,  # можно и не передавать, но так явнее
    )

    # Оценка качества
    metrics, val_pred_df = evaluate_on_split_store_level(
        preds_df,
        val_df_top,
        split_name=split_name,
    )

    # Сохраняем
    sarima_metrics_by_split[split_name] = metrics
    sarima_preds_by_split[split_name] = val_pred_df


2025-11-16 21:55:53,718 - sarima_store_level - INFO - auto_arima fitted: order=(2, 1, 0), seasonal_order=(1, 0, 0, 7)
2025-11-16 21:55:53,726 - sarima_store_level - INFO - [PID=10434] SARIMA OK for (store=48, family=PERSONAL CARE), horizon=16
2025-11-16 21:55:53,733 - sarima_store_level - INFO - [split_3] Pair (store=48, family=PERSONAL CARE) finished in 1129.86 seconds (queue + fit+forecast)
2025-11-16 21:55:53,799 - sarima_store_level - INFO - [PID=10434] start (store=48, family=MEATS)
2025-11-16 21:55:58,882 - sarima_store_level - INFO - auto_arima fitted: order=(2, 1, 0), seasonal_order=(1, 0, 1, 7)
2025-11-16 21:55:58,891 - sarima_store_level - INFO - [PID=10437] SARIMA OK for (store=48, family=POULTRY), horizon=16
2025-11-16 21:55:58,894 - sarima_store_level - INFO - [split_3] Pair (store=48, family=POULTRY) finished in 1135.02 seconds (queue + fit+forecast)
2025-11-16 21:55:58,967 - sarima_store_level - INFO - [PID=10437] start (store=48, family=HOME CARE)
2025-11-16 21:56:06,12

In [ ]:
# Преобразуем словарь метрик в DataFrame
metrics_rows = []
for split_name, m in sarima_metrics_by_split.items():
    row = {"split": split_name}
    row.update({
        "RMSLE": m["RMSLE"],
        "RMSE": m["RMSE"],
        "MAE": m["MAE"],
        "WAPE": m["WAPE"],
    })
    metrics_rows.append(row)

sarima_metrics_df = pd.DataFrame(metrics_rows).set_index("split")

# Добавим строку со средними по всем сплитам
mean_row = sarima_metrics_df.mean(axis=0).to_dict()
mean_row = pd.DataFrame(mean_row, index=["mean"])
sarima_metrics_df = pd.concat([sarima_metrics_df, mean_row])

sarima_metrics_df

In [ ]:
# Сохраняем метрики
metrics_path = "../data/experiment_info/sarima_store_metrics_by_split.csv"
sarima_metrics_df.to_csv(metrics_path, index=True)
logger.info(f"SARIMA metrics saved to {metrics_path}")

# Сохраняем предсказания по сплитам 
import pickle

preds_path = "../data/experiment_info/sarima_store_preds_by_split.pkl"
with open(preds_path, "wb") as f:
    pickle.dump(sarima_preds_by_split, f)

logger.info(f"SARIMA predictions by split saved to {preds_path}")

In [ ]:
sarima_metrics_by_split